# 가격 분포 검증 워크스루 (CH015)

`validation/PRICE/pricing-histogram-check/` 의 모듈을 **단계별로 돌려 보며 변수를 조절**하는 노트북.

우리가 만들려는 것은 판매자 가격 입력 화면에 띄울 **웹검색 시세 분포 그래프**다.
검증해야 할 질문은 두 가지다.

| 출력 | 질문 | 관문 지표 |
| --- | --- | --- |
| **A** 가격 범주별 매물 수 | 이 히스토그램이 *시장*을 그리는가, *우리 검색 편향*을 그리는가 | 가격 중립성 |
| **B** 상태별 가격 | 등급이 오르면 가격도 오르는가 | 단조성(Spearman) |

> ⛔ **기본 백엔드는 `mock`이다.** GMS 실호출은 마지막 셀에서 사용자가 직접 켠다
> (`AI/CLAUDE.md` 하드 룰 — 검색 라운드마다 과금).

설계: `.claude/plans/ai/CH015-comps-price-distribution.md`

In [ ]:
import sys
from pathlib import Path


def _find_spike_dir() -> Path:
    """노트북을 AI/ 에서 열든 스파이크 폴더에서 열든 모듈을 찾는다."""
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        if base.name == "pricing-histogram-check" and (base / "harvest.py").exists():
            return base
        candidate = base / "validation" / "PRICE" / "pricing-histogram-check"
        if candidate.exists():
            return candidate
    raise RuntimeError("pricing-histogram-check 폴더를 찾지 못했습니다")


SPIKE_DIR = _find_spike_dir()
if str(SPIKE_DIR) not in sys.path:
    sys.path.insert(0, str(SPIKE_DIR))

import harvest as hv
import hist_metrics as hm
import histogram_registry as reg
import run_histogram_sweep as sweep

print("스파이크 경로:", SPIKE_DIR)


def bars(buckets, width=40):
    """텍스트 막대그래프. 외부 라이브러리 없이 분포 모양을 본다."""
    peak = max((b.count for b in buckets), default=0)
    for b in buckets:
        filled = 0 if peak == 0 else round(b.count * width / peak)
        print(f"  {b.min:>9,}~{b.max:>9,}원 |{'#' * filled:<{width}}| {b.count}")

## 조절 변수

여기만 바꿔 가며 아래 셀들을 다시 돌리면 된다. **각 변수가 어느 축을 움직이는지**를 같이 적어 둔다.

In [ ]:
# --- 수집 레버 -------------------------------------------------------------
ROUND = "R3"  # R0 베이스라인 / R1 무비용 / R2 프롬프트 / R3 출력A 층화 / B 출력B / S1 카운트
LIMIT = 3  # 사용할 쿼리 수 (전체는 reg.QUERIES 10건)
MAX_COMPS = 30  # 수확 상한. 히스토그램 목표(12)보다 커야 의미가 있다
REPEAT = 2  # 축 B 재현성 반복 횟수

# --- mock 백엔드 성질 -------------------------------------------------------
POOL_SIZE = 40  # 검색이 물어 오는 매물 풀 크기
RANK_BIAS = 0.7  # 0=순위 무작위, 1=표준적 매물일수록 상위. **출력 A의 핵심 변수**

# --- 심사·판정 임계 ---------------------------------------------------------
MAD_THRESHOLD = hm.DEFAULT_MAD_THRESHOLD  # 이상치 컷 (기본 3.5)
RETAIL_PRICE = 120_000  # 신품 정가 앵커. None이면 정가 게이트 비활성
NEUTRALITY_ALERT = sweep.NEUTRALITY_ALERT  # 가격 중립성 경보선 (기본 0.15)
HIST_MIN = reg.HIST_MIN_SAMPLES  # 히스토그램을 그려도 되는 최소 표본 (기본 12)

# --- 예시 상품 --------------------------------------------------------------
PRODUCT = "501 빅E 데님"
BRAND = "리바이스"
CATEGORY = "팬츠"

config = reg.round_config(ROUND, repeats=REPEAT, max_comps=MAX_COMPS)
backend = hv.MockSearchBackend(pool_size=POOL_SIZE, rank_bias=RANK_BIAS)
print(f"라운드 {config.name} — {config.note}")
print(f"적용 레버: {', '.join(config.levers) or '(없음)'}")

## Step 1 — 쿼리를 어떻게 나눠 던지는가

레버마다 **던지는 쿼리 자체가 달라진다.** 검색 호출 수가 곧 비용이므로 여기서 몇 개가 생기는지가
축 C를 결정한다.

- **S5 특이도 사다리** — 좁은 것부터. 어느 단에서 멈췄는지가 `비교 수준`이 된다
- **S2 가격 층화** — top-k 절단이 전역이 아니라 층 안에서 일어나게 한다 (출력 A)
- **S3 상태축 층화** — 등급별 표본을 보장한다 (출력 B)
- **S1 카운트** — 매물이 아니라 *개수*를 묻는다

In [ ]:
print("[S5] 특이도 사다리")
for level, q in hv.build_specificity_ladder(PRODUCT, BRAND, CATEGORY):
    print(f"  {level:10s} → {q}")

print("\n[S2] 가격 층화 (출력 A)")
for label, q in hv.build_stratum_queries(f"{BRAND} {PRODUCT}"):
    print(f"  {label:15s} → {q}")

print("\n[S3] 상태축 층화 (출력 B)")
for grade, q in hv.build_condition_queries(f"{BRAND} {PRODUCT}"):
    print(f"  {grade} → {q}")

print("\n[S1] 카운트 쿼리 — 프롬프트가 요구하는 정밀도가 낮다")
low, high = reg.PRICE_STRATA[1]
print(hv.count_prompt(f"{BRAND} {PRODUCT}", low, high))

## Step 2 — 수집 (축 A·C)

`n_raw`(중복 포함) → `n_dedup`(중복 제거) → `n_valid`(심사 통과)로 줄어든다.
**`hist_ok`(n ≥ 12)가 주 판정 지표**다 — 표본이 한 자릿수인 히스토그램은 없느니만 못하다.

In [ ]:
outcome = hv.harvest(
    backend, config, f"{BRAND} {PRODUCT}", retail_price=RETAIL_PRICE, brand=BRAND, category=CATEGORY
)

print(f"쿼리      : {outcome.query}")
print(f"비교 수준 : {outcome.level or '-'}")
print(f"n_raw {outcome.n_raw} → n_dedup {outcome.n_dedup} → n_valid {outcome.n_valid}")
print(f"검색 호출 : {outcome.search_calls}회  (축 C — 이게 곧 비용)")
print(f"hist_ok   : {outcome.n_valid >= HIST_MIN}  (기준 n≥{HIST_MIN})")
for note in outcome.notes[:5]:
    print("  note:", note)

## Step 3 — 사례 적격성 심사 (M1 ②)

거래사례비교법 6단계 중 **지금 프로덕션에 없는 단계**. 비용 0인데 빠져 있다.

- **S6 정가 게이트** — 정가 대비 비율이 상식 범위 밖이면 다른 상품이 섞인 것
- **L5 MAD 이상치** — 로그 스케일 modified z-score. 현행 "양끝 1건 trim"은 이상치가 2건이면 못 걷어낸다

`MAD_THRESHOLD`를 낮추면 더 공격적으로 잘라낸다.

In [ ]:
raw = hv.harvest(hv.MockSearchBackend(pool_size=POOL_SIZE, rank_bias=RANK_BIAS),
                 reg.round_config("R0", max_comps=MAX_COMPS), f"{BRAND} {PRODUCT}")
prices = [i.price for i in raw.items]

gate = hv.retail_gate_mask(raw.items, RETAIL_PRICE)
kept_after_gate = [p for p, drop in zip(prices, gate, strict=True) if not drop]
flags = hm.outlier_mask(kept_after_gate, threshold=MAD_THRESHOLD)
survivors = [p for p, bad in zip(kept_after_gate, flags, strict=True) if not bad]

print(f"수집 {len(prices)}건")
print(f"  정가 게이트({RETAIL_PRICE:,}원 기준) 탈락 {sum(gate)}건 → {len(kept_after_gate)}건")
print(f"  MAD 이상치(|z|>{MAD_THRESHOLD}) 탈락 {sum(flags)}건 → {len(survivors)}건")
print(f"\n중앙값 {hm.median(prices):,.0f} → 심사 후 {hm.median(survivors):,.0f}")
print(f"최댓값 {max(prices):,} → 심사 후 {max(survivors):,}")

## Step 4 — 분포 추정 (M4)

중고 가격은 **로그정규**에 가깝다(하한 0, 우측 꼬리 김). 그래서 등간격이 아니라 로그 스케일로 자르고,
버킷 개수는 **Freedman–Diaconis** 규칙(`2·IQR/n^(1/3)`)이 정한다 — "왜 5개냐"에 답이 되는 부분.

`MIN/MAX_BUCKETS`를 바꿔 해상도를 조절해 볼 수 있다.

In [ ]:
MIN_BUCKETS, MAX_BUCKETS = 3, 12

edges = hm.log_bucket_edges(survivors, min_buckets=MIN_BUCKETS, max_buckets=MAX_BUCKETS)
buckets = hm.histogram(survivors, edges)

print(f"FD 폭(로그) {hm.freedman_diaconis_width([__import__('math').log(p) for p in survivors]):.4f}")
print(f"버킷 {len(buckets)}개 / 균일도 {hm.bucket_uniformity(buckets):.2f}\n")
bars(buckets)
print(f"\n합계 {sum(b.count for b in buckets)}건 (표본 {len(survivors)}건과 일치해야 한다)")

## Step 5 — 불확실성 (M6)

현행 `confidence`는 임의 가중합(`0.5/0.2/0.3`)이라 "0.62가 무슨 뜻이냐"에 답할 수 없다.
**부트스트랩 신뢰구간**은 분포 가정이 없고 **API 호출이 0건**이며, CI 폭이 곧 해석 가능한 신뢰도다.

아래에서 표본 크기를 줄여 가며 CI가 어떻게 넓어지는지 본다 — 폴백 사다리의 근거다.

In [ ]:
import random
import statistics

interval = hm.bootstrap_ci([float(p) for p in survivors])
print(f"중앙값 {interval.point:,.0f}원 (95% CI {interval.low:,.0f}~{interval.high:,.0f}, "
      f"폭 {interval.width_ratio:.0%})\n")

# ⚠️ `survivors[:n]` 로 자르면 안 된다 — 앞쪽은 순위 편향으로 중앙값 근처에 몰려 있어서
#    표본이 작을수록 CI가 오히려 좁아 보인다. 무작위 부분표본을 여러 번 뽑아 평균낸다.
rng = random.Random(reg.SEED)
TRIALS = 15

print("표본 크기별 CI 폭 (무작위 부분표본 15회 평균) — 작을수록 넓어진다")
for n in (3, 5, 8, 12, 20, len(survivors)):
    if n > len(survivors):
        continue
    widths = [
        hm.bootstrap_ci([float(p) for p in rng.sample(survivors, n)], iterations=400).width_ratio
        for _ in range(TRIALS)
    ]
    verdict = "히스토그램" if n >= HIST_MIN else "range만"
    print(f"  n={n:3d}  폭 {statistics.mean(widths):5.0%}  → {verdict}")

## Step 6 — 가격 중립성 ★ (축 D — **출력 A 채택의 관문**)

"매물 수"는 검색이 가격에 대해 중립적일 때만 의미가 있다. 그런데 LLM 웹검색은 *관련도 순위* 검색이고,
거기에 상한(`max_comps`)으로 **순위 리스트를 자르면 분포가 구조적으로 왜곡된다.**

`RANK_BIAS`를 0 → 1로 올리며 지표가 반응하는지 본다. 반응하지 않으면 지표가 다른 걸 재고 있는 것이다.

> **표집 잡음을 먼저 빼야 한다.** n건에서 k건만 뽑는 것 자체가 거리를 만들어서, 순위가 완전히
> 무작위여도 원시 Wasserstein은 0이 아니다. 실제로 초기 구현이 그랬고 `rank_bias` 0과 0.95가
> 구분되지 않았다. 그래서 `price_neutrality`는 **같은 표본에서 무작위로 k건 뽑은 귀무분포를 빼고
> 초과분만** 돌려주는 순열검정 통계량이다 — **0이 중립**이다.

In [ ]:
import statistics

R0_CFG = reg.round_config("R0", max_comps=MAX_COMPS)
PROBE_QUERIES = reg.QUERIES[:5]  # 쿼리 1건이면 유한표본 잡음에 묻힌다


def neutrality_at(bias):
    return statistics.mean(
        sweep.price_neutrality(
            hv.harvest(hv.MockSearchBackend(pool_size=POOL_SIZE, rank_bias=bias), R0_CFG, q).items,
            k=10,
        )
        for q in PROBE_QUERIES
    )


print(f"경보선 {NEUTRALITY_ALERT} (초과분 기준 — 0이 중립)\n")
print("rank_bias   초과분    판정")
for bias in (0.0, 0.25, 0.5, 0.75, 0.95):
    score = neutrality_at(bias)
    mark = "!! 편향" if score > NEUTRALITY_ALERT else "OK"
    print(f"  {bias:4.2f}     {score:+.4f}   {mark}")

print("\n편향을 올리면 초과분이 따라 오른다 — 지표가 순위 편향을 실제로 재고 있다는 뜻.")
print("무작위(0.00) 근처에서 ±0.03 정도가 남는 것은 쿼리당 관측 순서가 1개뿐이라 그렇다.")

## Step 7 — 등급 단조성 (축 D — **출력 B**)

Spearman이 쓰이는 **유일한 자리**다. 상품 간 비교에는 쓰지 않는다 — 개별 중고 의류에
"알려진 참 가격 순서"가 없기 때문(정답 부재).

단조성이 깨지면 결론은 *"가격이 틀렸다"*가 아니라 **"우리 상태 신호가 잡음이다"** 이다.

> ⚠️ mock은 등급을 가격과 무관하게 순환 배정한다 → 여기서 ρ가 낮은 것은 **음성 대조**이지 발견이 아니다.

In [ ]:
b_outcome = hv.harvest(backend, reg.round_config("B", max_comps=MAX_COMPS), f"{BRAND} {PRODUCT}")
rho, medians = sweep.grade_monotonicity(b_outcome.items)

print("등급별 중앙값 (나쁨 → 좋음 순으로 올라야 한다)")
for grade in ("C", "B", "A", "S"):
    if grade in medians:
        print(f"  {grade}  {medians[grade]:>9,}원")
print(f"\nSpearman rho = {rho}")
print("판정:", "단조성 성립" if rho >= 0.5 else "상태 신호가 잡음 — 백분위 등급 밴드의 전제가 깨진다")

## Step 8 — 재현성 (축 B)

정답이 없는 문제에서 쓸 수 있는 정확도 프록시. 같은 쿼리를 반복해 분포가 얼마나 흔들리는지 본다
(재검사 신뢰도). 거리는 버킷 경계에 의존하는 L1 대신 **Wasserstein**을 쓴다.

> ⚠️ mock은 결정적이라 **항상 0.0000**이 나온다. 이 축은 실백엔드에서만 의미가 있다.

In [ ]:
runs = [
    [i.price for i in hv.harvest(backend, config, f"{BRAND} {PRODUCT}").items]
    for _ in range(REPEAT)
]
print(f"반복 {REPEAT}회 — 정규화 Wasserstein 평균 {sweep.reproducibility(runs):.4f}")
print("(mock은 결정적이라 0. 실백엔드에서는 캐시를 우회해야 의미가 있다)")

shifted = [p * 1.1 for p in runs[0]]
print(f"\n참고 — 분포를 10% 옮기면: {hm.normalized_wasserstein(runs[0], shifted):.4f}")

## Step 9 — 라운드 비교

레버 조합을 격자로 돌리지 않는다(검색 라운드가 곧 비용). R0 → R1 → R2 → R3 순으로 올리다
`hist_ok`가 목표에 닿으면 **거기서 멈춘다.**

In [ ]:
queries = reg.QUERIES[:LIMIT]

print(f"{'라운드':<6}{'검색/쿼리':>10}{'n_valid':>9}{'hist_ok':>9}  비고")
for key in ("R0", "R1", "R2", "R3", "B"):
    cfg = reg.round_config(key, max_comps=MAX_COMPS)
    results = [hv.harvest(backend, cfg, q, retail_price=RETAIL_PRICE) for q in queries]
    calls = sum(r.search_calls for r in results) / len(results)
    valid = sum(r.n_valid for r in results) / len(results)
    ok = sum(1 for r in results if r.n_valid >= HIST_MIN)
    print(f"{key:<6}{calls:>10.1f}{valid:>9.1f}{ok:>6}/{len(results)}  {cfg.note[:34]}")

## Step 10 — S1 카운트 쿼리 (최우선 선결 확인)

출력 A가 필요한 건 매물 목록이 아니라 **구간별 개수**다. 순서를 뒤집으면
**top-k 절단 편향이 원천적으로 사라진다** — 개수는 자르는 대상이 아니다.

실측에서 가장 먼저 확인할 것: 검색 그라운딩이 플랫폼의 가격 필터별 "총 N개"를 실제로 읽어 오는가.

In [ ]:
s1 = hv.harvest(backend, reg.round_config("S1"), f"{BRAND} {PRODUCT}")
print(f"검색 호출 {s1.search_calls}회 (구간 수와 같아야 한다)\n")

peak = max(s1.counts_by_stratum.values(), default=1)
for label, count in s1.counts_by_stratum.items():
    print(f"  {label:>15} |{'#' * round(count * 30 / peak):<30}| {count}")
print("\n매물을 한 건도 파싱하지 않고 히스토그램이 나온다 —— 요구 정밀도가 낮다.")

## Step 11 — ⛔ 실호출 (사용자 전용)

**아래 셀은 기본으로 꺼져 있다.** `RUN_REAL = True`로 바꾸는 것은 사용자만 한다 —
GMS 웹검색은 팀 공용 유료 쿼터이고 검색 라운드마다 과금된다(`AI/CLAUDE.md` 하드 룰).

권장 순서: **S1 먼저**(3쿼리 × 5구간 = 15회, 가장 쌈) → 결과에 따라 R0 → R1 → …

In [ ]:
RUN_REAL = False  # ← 사용자만 True 로 바꾼다
REAL_ROUND = "S1"
REAL_LIMIT = 3

if not RUN_REAL:
    est = len(reg.PRICE_STRATA) * REAL_LIMIT if REAL_ROUND == "S1" else REAL_LIMIT
    print(f"실호출 비활성. RUN_REAL=True 로 바꾸면 약 {est}회 검색이 발생합니다.")
else:
    real_cfg = reg.round_config(REAL_ROUND, max_comps=MAX_COMPS)
    real_backend = hv.GeminiSearchBackend(
        reg.DEFAULT_MODELS["gemini"], follow_pause_turn=real_cfg.follow_pause_turn
    )
    reports = sweep.run_round(real_backend, real_cfg, reg.QUERIES[:REAL_LIMIT])
    sweep.print_report(real_cfg, reports)

## 정리 — 이 노트북으로 말할 수 있는 것 / 없는 것

**말할 수 있는 것**
- 지표가 작동한다 — `RANK_BIAS`를 올리면 가격 중립성이 반응한다(Step 6)
- 심사·버킷·CI 계산 경로가 끝까지 돈다(Step 3~5)
- 레버별 검색 호출 수 = 비용 구조(Step 9)

**말할 수 없는 것**
- 실제 수확량 n — mock은 `POOL_SIZE`만큼 항상 준다. 실측은 평균 1.7건이었다(LAB-47)
- 재현성 — mock은 결정적이라 항상 0
- 등급 단조성 — mock은 등급을 무작위 배정한다
- **comps의 가격·URL이 실제로 맞는지** — 풀링 적합성 판정(`--pool-out`)의 몫

**다음 단계**: Step 11에서 `S1`을 실호출로 돌려 카운트 쿼리가 성립하는지부터 확인한다.